# Class 03 · Part 2: Using gridded data

In part 1, `class03_get_gridded_data.ipynb`, you downloaded the E-OBS daily mean temperature and looked
inside the netCDF file. Now you

1. **select** days, months, years and places,
2. draw a **map of one month** and a **map of one year**,
3. compute the **temperature of Europe** as a time series, with the right weights, and
4. save the three plots for your page.

The exercise: the same for precipitation.

## 1. Packages and data

`cartopy` (short `ccrs` for its map projections) draws coastlines and maps, as in class 01.

In [ ]:
import cartopy.crs as ccrs
import matplotlib.pyplot as plt
import numpy as np
import xarray as xr

from utils import download_eobs

ds = xr.open_dataset(download_eobs("tg"), chunks={"time": 31})
tg = ds["tg"]
tg

## 2. Selecting

`.sel` selects by the values of the coordinates. Dates can be a day, a month or a year:

In [ ]:
one_day = tg.sel(time="2025-07-15")
one_month = tg.sel(time="2025-07")  # all days of July 2025
one_year = tg.sel(time="2024")  # all days of 2024

print(one_day.shape, one_month.shape, one_year.shape)  # (days,) latitudes, longitudes

A place: the cell **nearest** to a latitude and longitude, e.g. Hamburg. That gives a time series, like the
station of class 02, but from the grid.

In [ ]:
hamburg = tg.sel(latitude=53.55, longitude=10.0, method="nearest")

fig, ax = plt.subplots(figsize=(10, 3))
hamburg.resample(time="MS").mean().plot(ax=ax)
ax.set_xlabel("year")
ax.set_title("Hamburg, monthly mean temperature from E-OBS")
plt.show()

## 3. A map of one month

The mean over the days of July: `.mean("time")` averages along the dimension `time`, and leaves a map.

A **projection** is the way to draw the round Earth on flat paper. `LambertConformal` is a good one for
Europe. `transform=ccrs.PlateCarree()` tells cartopy that our data is on a plain lon-lat grid.

In [ ]:
july = tg.sel(time="2025-07").mean("time")

fig = plt.figure(figsize=(8, 6))
ax = plt.axes(
    projection=ccrs.LambertConformal(central_longitude=10, central_latitude=52)
)
july.plot(
    ax=ax,
    transform=ccrs.PlateCarree(),
    cmap="RdYlBu_r",
    cbar_kwargs={"label": "temperature / °C"},
)
ax.coastlines(linewidth=0.5)
ax.set_extent([-25, 45, 34, 72], crs=ccrs.PlateCarree())  # west, east, south, north
ax.set_title("Mean temperature, July 2025 (E-OBS)")

fig.savefig(
    "/app/pages/03_gridded_data/temperature_month.png", dpi=100, bbox_inches="tight"
)
plt.show()

**Try it:** choose another month, e.g. `"2025-01"`, and run the cell again. Change the title too.

## 4. A map of one year

The same, with all days of one year:

In [ ]:
year = tg.sel(time="2024").mean("time")

fig = plt.figure(figsize=(8, 6))
ax = plt.axes(
    projection=ccrs.LambertConformal(central_longitude=10, central_latitude=52)
)
year.plot(
    ax=ax,
    transform=ccrs.PlateCarree(),
    cmap="RdYlBu_r",
    cbar_kwargs={"label": "temperature / °C"},
)
ax.coastlines(linewidth=0.5)
ax.set_extent([-25, 45, 34, 72], crs=ccrs.PlateCarree())
ax.set_title("Mean temperature, 2024 (E-OBS)")

fig.savefig(
    "/app/pages/03_gridded_data/temperature_year.png", dpi=100, bbox_inches="tight"
)
plt.show()

## 5. The temperature of Europe

One value for all of Europe and every day: the **mean over all cells**. First we cut out Europe with
`slice(from, to)`:

In [ ]:
europe = tg.sel(latitude=slice(35, 72), longitude=slice(-25, 45))

**But the cells are not all the same size.** A cell is always 0.25° wide, but the lines of longitude come
closer towards the North Pole, so a cell in Norway is much narrower than a cell in Spain:

In [ ]:
for latitude in [35, 50, 70]:
    width = (
        0.25 * 111.32 * np.cos(np.deg2rad(latitude))
    )  # 1° of longitude is 111.32 km at the equator
    print(f"at {latitude}°N a cell is {width:.1f} km wide")

A plain mean would count the small northern cells as much as the large southern ones. So every cell gets
a **weight**: its relative area, the cosine of its latitude. `.weighted(weights).mean(...)` takes care of it,
and of the cells without values (the sea).

`.compute()` does the work now and keeps the result (all days of 15 years: it takes a few seconds).

In [ ]:
weights = np.cos(np.deg2rad(europe.latitude))

europe_daily = europe.weighted(weights).mean(("latitude", "longitude")).compute()
without_weights = europe.mean(("latitude", "longitude")).compute()

print(f"mean with weights:    {float(europe_daily.mean()):.2f} °C")
print(f"mean without weights: {float(without_weights.mean()):.2f} °C")

Almost 1 °C difference: without weights, the cold north counts too much.

Now the time series, monthly and yearly, as in class 02:

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
europe_daily.resample(time="MS").mean().plot(
    ax=ax, color="lightgray", label="monthly mean"
)
europe_daily.resample(time="YS").mean().plot(
    ax=ax, color="firebrick", linewidth=2, marker="o", label="yearly mean"
)
ax.set_ylabel("temperature / °C")
ax.set_xlabel("year")
ax.set_title("Mean temperature of Europe (35–72°N, 25°W–45°E), E-OBS")
ax.legend()

fig.savefig(
    "/app/pages/03_gridded_data/temperature_europe.png", dpi=100, bbox_inches="tight"
)
plt.show()

## 6. Exercise: precipitation

Do the same for the precipitation, `"rr"` (in mm), which you downloaded in part 1:

1. Open it, and take its variable `rr`.
2. A map of the precipitation of **one month**, saved as `/app/pages/03_gridded_data/precipitation_month.png`.
3. A map of **one year**, saved as `/app/pages/03_gridded_data/precipitation_year.png`.
4. The precipitation of **Europe** as a time series, saved as `/app/pages/03_gridded_data/precipitation_europe.png`.

Hints:

- Precipitation adds up: use `.sum("time", min_count=1)` for the maps instead of `.mean("time")`.
  `min_count=1` keeps the sea empty; without it, the sum over no values is 0 and the sea looks dry.
- For Europe: first the weighted **mean** over the cells (as in section 5), then the monthly and yearly
  **sums** with `.resample(...).sum()`.
- A colour map for precipitation: `cmap="Blues"`.

In [ ]:
# 1. open the precipitation

In [ ]:
# 2. a map of one month

In [ ]:
# 3. a map of one year

In [ ]:
# 4. the precipitation of Europe

## 7. Your page

Open `pages/03_gridded_data/my_page.rst` in JupyterLab, write your name, and describe your maps and time
series in a few sentences. Look at it on http://localhost:8000 → *Classes → 03*.